# Re-evaluation — is the model bad, or is the measurement bad?

The first full run reported F0.5 0.269 for the fine-tune vs 0.285 for coedit-large.
This notebook re-scores the **same saved checkpoint** without retraining, fixing
four defects in the original evaluation:

1. **Biased comparison.** Gold labels come from `derive_edits`. The baseline's
   output was passed through that same function, so it inherited difflib's
   segmentation for free; the model had to guess it. Both sides are now
   canonicalised the same way.
2. **Prompt mismatch.** LLaMA-Factory joins instruction and input with `"\n"`
   (`converter.py:100`), but the original eval asked with `"\n\n"`.
3. **Wrong domain.** The old eval set was ~95% general-domain web text. A
   banking set is now built explicitly.
4. **No text-level metric.** Nothing measured whether applying the edits
   actually reproduces the target sentence.

The model still emits JSON edits — that contract is unchanged.

In [ ]:
import os, re, json, random, difflib, unicodedata, time, glob
from pathlib import Path
import pandas as pd
import torch

random.seed(13)
t0 = time.time()
def stage(m): print(f"[stage +{time.time()-t0:7.1f}s] {m}", flush=True)

# The previous kernel's output is attached as a data source.
def find_dir(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

MODEL_DIR = find_dir("/kaggle/input/**/gec_bank/saves/qwen3-0.6b-bankgec")
DEV_DIR   = find_dir("/kaggle/input/**/gec_bank/data/bank_gec_dev.parquet")
CFPB_CSV  = find_dir("/kaggle/input/**/complaints.csv")

print("model:", MODEL_DIR)
print("dev  :", DEV_DIR)
print("cfpb :", CFPB_CSV)
assert MODEL_DIR and DEV_DIR, "attach the training kernel's output as a data source"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Shared logic — copied verbatim from the training notebook

In [ ]:
PROTECTED_TERMS = {
    # instruments & identifiers
    "IBAN", "SWIFT", "BIC", "ACH", "SEPA", "IFSC", "MICR", "CUSIP", "ISIN", "LEI",
    "APR", "APY", "EAR", "LIBOR", "SOFR", "EURIBOR", "CBE", "FRB", "FDIC", "OCC",
    "KYC", "AML", "CDD", "EDD", "PEP", "SAR", "CTR", "UDAAP", "TILA", "RESPA",
    "ACH", "POS", "ATM", "PIN", "CVV", "OTP", "SLA", "NDA", "PSD2", "GDPR",
    "LTV", "DTI", "FICO", "NPV", "IRR", "EBITDA", "YTD", "QoQ", "YoY",
    "Basel", "IFRS", "GAAP", "CRS", "FATCA",
    # legalese that reads like an error but is not
    "hereinafter", "heretofore", "thereunder", "thereof", "herein", "hereby",
    "notwithstanding", "aforementioned", "whereas", "pursuant", "thereto",
    "indemnify", "indemnification", "forbearance", "novation", "subrogation",
    "escrow", "lien", "encumbrance", "covenant", "accrual", "amortization",
    "per annum", "pro rata", "force majeure", "bona fide", "inter alia",
}

# Regexes for things that must survive verbatim
PROTECTED_PATTERNS = [
    re.compile(r"\b[A-Z]{2,}\b"),                              # any all-caps acronym
    re.compile(r"[$€£¥]\s?[\d,]+(?:\.\d+)?"),                  # $1,250.00
    re.compile(r"\b(?:USD|EUR|GBP|EGP|AED|SAR|JPY)\s?[\d,]+"),  # EGP 1,250,000
    re.compile(r"\b\d+(?:\.\d+)?\s?%"),                       # 4.75 %
    re.compile(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b"),                # dates
    re.compile(r"\bSection\s+\d+"),                            # Section 4.2
    re.compile(r"\bClause\s+\d+"),
    re.compile(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\s+Agreement\b"),  # defined terms
]

def strip_punct(tok):
    """Split a whitespace token into (lead_punct, core, trail_punct)."""
    m = re.match(r"^([^\w]*)(.*?)([^\w]*)$", tok, flags=re.UNICODE)
    return m.group(1), m.group(2), m.group(3)

# multi-word terms need span patterns, or the corruptor will split them
# ("per annum" -> "per the annum"). Auto-generated so the term list stays simple.
PROTECTED_PATTERNS += [
    re.compile(r"\b" + re.escape(t) + r"\b", re.I)
    for t in PROTECTED_TERMS if " " in t
]

_PROTECTED_LOWER = {t.lower() for t in PROTECTED_TERMS}

def is_protected(token, sentence=None):
    _, core, _ = strip_punct(token)
    if not core:
        return True
    if core.lower() in _PROTECTED_LOWER:      # case-insensitive: "Pursuant" == "pursuant"
        return True
    if core.isupper() and len(core) >= 2:
        return True
    if any(ch.isdigit() for ch in core):
        return True
    if sentence:
        for pat in PROTECTED_PATTERNS:
            for m in pat.finditer(sentence):
                if core in m.group(0):
                    return True
    return False

def free(tokens, i, sentence):
    """Safe to insert *after* token i? Must not split a protected multi-token span."""
    if is_protected(tokens[i], sentence):
        return False
    return i + 1 >= len(tokens) or not is_protected(tokens[i + 1], sentence)

print(f"{len(PROTECTED_TERMS)} protected terms, {len(PROTECTED_PATTERNS)} patterns")

In [ ]:
WORD_RE = re.compile(r"\S+")

def toks(s):
    return WORD_RE.findall(s)

def derive_edits(src, tgt):
    """Word-level diff -> compact edit list.

    Occurrence index `n` is counted in *substring* space, not word space, because
    apply_edits resolves spans with str.find(). Getting this wrong silently breaks
    any sentence containing a repeated word ("the" also occurs inside "there").
    """
    spans = [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(src)]
    a = [s[0] for s in spans]
    b = toks(tgt)
    edits = []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(a=a, b=b, autojunk=False).get_opcodes():
        if tag == "equal":
            continue
        if tag == "insert":
            # anchor to an adjacent word: span anchors survive tokenizer drift,
            # integer positions do not
            if i1 > 0:
                k, pre = i1 - 1, True
            elif i1 < len(a):
                k, pre = i1, False
            else:
                return None
            cs, ce = spans[k][1], spans[k][2]
            o = src[cs:ce]
            ins = " ".join(b[j1:j2])
            c = (o + " " + ins) if pre else (ins + " " + o)
        else:
            cs, ce = spans[i1][1], spans[i2 - 1][2]
            o = src[cs:ce]
            c = " ".join(b[j1:j2])
        edits.append({"o": o, "c": c, "n": src[:cs].count(o) + 1})
    return edits or None

def apply_edits(src, edits):
    """Inverse of derive_edits. Used to validate every training pair round-trips."""
    out = src
    for e in edits:
        o, c, n = e["o"], e["c"], e.get("n", 1)
        idx, start = -1, 0
        for _ in range(n):
            idx = out.find(o, start)
            if idx == -1:
                return None
            start = idx + 1
        out = out[:idx] + c + out[idx + len(o):]
    return re.sub(r"\s+", " ", out).strip()

def roundtrips(src, tgt):
    e = derive_edits(src, tgt)
    if e is None:
        return None
    got = apply_edits(src, e)
    norm = lambda s: re.sub(r"\s+", " ", s).strip()
    return e if got is not None and norm(got) == norm(tgt) else None

# sanity check
_s = "The account holder were notified of the changes to there statement."
_t = "The account holder was notified of the changes to their statement."
print(json.dumps(roundtrips(_s, _t), ensure_ascii=False))
print(apply_edits(_s, roundtrips(_s, _t)))

In [ ]:
INSTRUCTION = (
    "Correct grammar, spelling, and punctuation. Do not change meaning, tone, "
    "financial terminology, defined terms, figures, or legal phrasing. "
    "Reply with a JSON array of edits, or [] if the text is correct."
)

def to_alpaca(src, edits, tag):
    return {
        "instruction": INSTRUCTION,
        "input": src,
        "output": json.dumps(edits, ensure_ascii=False, separators=(",", ":")),
        "_tag": tag,
    }

In [ ]:
AGREEMENT = {"is":"are","are":"is","was":"were","were":"was","has":"have","have":"has",
             "does":"do","do":"does","wasn't":"weren't","weren't":"wasn't",
             "isn't":"aren't","aren't":"isn't","hasn't":"haven't","haven't":"hasn't"}

CONFUSIONS = {"their":"there","there":"their","they're":"their","its":"it's","it's":"its",
              "then":"than","than":"then","affect":"effect","effect":"affect",
              "principal":"principle","principle":"principal","ensure":"insure",
              "advice":"advise","advise":"advice","accept":"except","except":"accept",
              "lose":"loose","loose":"lose","complement":"compliment",
              "your":"you're","you're":"your","to":"too","too":"to","whose":"who's"}

PREPS = ["of","for","to","in","on","at","with","from","by","about","under","within"]

TENSE = {"received":"receive","processed":"process","notified":"notify","applied":"apply",
         "charged":"charge","credited":"credit","approved":"approve","issued":"issue",
         "reviewed":"review","submitted":"submit","exceeded":"exceed","incurred":"incur",
         "took":"take","made":"make","paid":"pay","held":"hold","sent":"send","was":"is"}


def _swap(tokens, i, new_core, sentence):
    lead, core, trail = strip_punct(tokens[i])
    if not core or is_protected(tokens[i], sentence):
        return False
    if core[0].isupper():
        new_core = new_core[0].upper() + new_core[1:]
    tokens[i] = lead + new_core + trail
    return True


def err_agreement(tk, s):
    idx = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in AGREEMENT]
    if not idx: return False
    i = random.choice(idx)
    return _swap(tk, i, AGREEMENT[strip_punct(tk[i])[1].lower()], s)

def err_confusion(tk, s):
    idx = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in CONFUSIONS]
    if not idx: return False
    i = random.choice(idx)
    return _swap(tk, i, CONFUSIONS[strip_punct(tk[i])[1].lower()], s)

def err_preposition(tk, s):
    idx = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in PREPS]
    if not idx: return False
    i = random.choice(idx)
    cur = strip_punct(tk[i])[1].lower()
    return _swap(tk, i, random.choice([p for p in PREPS if p != cur]), s)

def err_tense(tk, s):
    idx = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in TENSE]
    if not idx: return False
    i = random.choice(idx)
    return _swap(tk, i, TENSE[strip_punct(tk[i])[1].lower()], s)

def err_article(tk, s):
    arts = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in ("a","an","the")]
    if arts and random.random() < 0.55:
        i = random.choice(arts)
        if is_protected(tk[i], s): return False
        if random.random() < 0.5:
            del tk[i]                                   # dropped article
        else:
            cur = strip_punct(tk[i])[1].lower()
            alt = {"a":"an","an":"a","the":"a"}[cur]
            return _swap(tk, i, alt, s)
        return True
    # spurious article
    cand = [i for i in range(1, len(tk)) if free(tk, i, s) and not is_protected(tk[i-1], s)]
    if not cand: return False
    tk.insert(random.choice(cand), "the")
    return True

def err_plural(tk, s):
    cand = [i for i, t in enumerate(tk)
            if len(strip_punct(t)[1]) > 3 and not is_protected(t, s)]
    if not cand: return False
    i = random.choice(cand)
    lead, core, trail = strip_punct(tk[i])
    new = core[:-1] if core.endswith("s") and not core.endswith("ss") else core + "s"
    tk[i] = lead + new + trail
    return True

def err_typo(tk, s):
    cand = [i for i, t in enumerate(tk)
            if len(strip_punct(t)[1]) >= 6 and not is_protected(t, s)]
    if not cand: return False
    i = random.choice(cand)
    lead, core, trail = strip_punct(tk[i])
    j = random.randrange(1, len(core) - 2)
    mode = random.random()
    if mode < 0.45:
        core = core[:j] + core[j+1] + core[j] + core[j+2:]      # transposition
    elif mode < 0.75:
        core = core[:j] + core[j] + core[j:]                    # doubled letter
    else:
        core = core[:j] + core[j+1:]                            # dropped letter
    tk[i] = lead + core + trail
    return True

def err_punct(tk, s):
    commas = [i for i, t in enumerate(tk) if t.endswith(",")]
    if commas and random.random() < 0.5:
        i = random.choice(commas)
        tk[i] = tk[i][:-1]                                      # dropped comma
        return True
    cand = [i for i in range(2, len(tk) - 2)
            if not tk[i].endswith((",", ".", ";")) and free(tk, i, s)]
    if not cand: return False
    i = random.choice(cand)
    tk[i] = tk[i] + ","                                         # spurious comma
    return True

def err_capital(tk, s):
    if len(tk) < 3: return False
    lead, core, trail = strip_punct(tk[0])
    if not core or not core[0].isupper() or is_protected(tk[0], s):
        return False
    tk[0] = lead + core[0].lower() + core[1:] + trail
    return True


ERRORS = [(err_agreement, 0.20), (err_confusion, 0.16), (err_preposition, 0.15),
          (err_article, 0.15), (err_tense, 0.11), (err_plural, 0.09),
          (err_typo, 0.07), (err_punct, 0.05), (err_capital, 0.02)]

def corrupt(sentence, max_errors=2):
    fns = [f for f, _ in ERRORS]
    wts = [w for _, w in ERRORS]
    tk = toks(sentence)
    applied = 0
    for _ in range(random.randint(1, max_errors) * 3):
        if applied >= max_errors:
            break
        fn = random.choices(fns, weights=wts, k=1)[0]
        if fn(tk, sentence):
            applied += 1
    out = " ".join(tk)
    return out if applied and out != sentence else None

## Scoring

`score_edits` is unchanged. What changes is *what we feed it*.

`canonical()` converts a system's edits into the same representation as the gold
labels: apply the edits, then re-derive them with `derive_edits`. The baseline
already got this for free; now the fine-tune does too. `exact()` asks the only
question that matters in production — does applying the edits reproduce the
target sentence?

In [ ]:
def score_edits(gold, pred):
    g = {(e["o"], e["c"]) for e in gold}
    p = {(e["o"], e["c"]) for e in pred}
    return len(g & p), len(p - g), len(g - p)

def prf(tp, fp, fn, beta=0.5):
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    b2 = beta ** 2
    F = ((1 + b2) * P * R / (b2 * P + R)) if (P and R) else 0.0
    return P, R, F

def norm(s):
    return re.sub(r"\s+", " ", s).strip()

def sane(src_text, edits):
    """Drop malformed edits and spans that are not actually in the source."""
    return [e for e in edits
            if isinstance(e, dict) and e.get("o") and e["o"] in src_text]

def canonical(src_text, edits):
    """Re-express edits in difflib's segmentation, the one the gold labels use."""
    edits = sane(src_text, edits)
    if not edits:
        return []
    fixed = apply_edits(src_text, edits)
    if fixed is None:
        return []
    return derive_edits(src_text, fixed) or []

def apply_edits_lenient(src_text, edits):
    """Apply what resolves, skip what does not.

    apply_edits is all-or-nothing: correct when building gold, wrong at
    inference. `n` is computed against the original sentence but every applied
    edit mutates the string, so a later index can stop resolving and take the
    whole list with it -- including edits that were perfectly good.
    """
    out, used = src_text, 0
    for e in sane(src_text, edits):
        o, c = e["o"], e.get("c", "")
        try:
            n = max(1, int(e.get("n", 1)))
        except (TypeError, ValueError):
            n = 1
        idx, start = -1, 0
        for _ in range(n):
            idx = out.find(o, start)
            if idx == -1:
                break
            start = idx + 1
        if idx == -1:
            idx = out.find(o)
            if idx == -1:
                continue
        out = out[:idx] + c + out[idx + len(o):]
        used += 1
    return norm(out), used


def applied_text(src_text, edits):
    return apply_edits_lenient(src_text, edits)[0]


def applied_text_strict(src_text, edits):
    """The old behaviour, kept so the fix can be quantified."""
    edits = sane(src_text, edits)
    if not edits:
        return norm(src_text)
    out = apply_edits(src_text, edits)
    return norm(out) if out is not None else norm(src_text)


def corrupt_one(sentence, fn, attempts=6):
    """Inject exactly one error of a known type, so recall can be attributed."""
    for _ in range(attempts):
        tk = toks(sentence)
        if fn(tk, sentence):
            out = " ".join(tk)
            if out != sentence:
                return out
    return None

def parse(raw):
    m = re.search(r"\[.*\]", raw, re.S)
    if not m:
        return [], False
    try:
        v = json.loads(m.group(0))
        return (v, True) if isinstance(v, list) else ([], False)
    except json.JSONDecodeError:
        return [], False

## Eval sets — banking measured separately from general

In [ ]:
stage("building banking eval set")

# The old eval set was ~95% general web text. Build a genuinely banking one from
# CFPB using the same cleaning rules the Spark stage used, then inject errors
# with the same corrupt() the training data used, so gold is exact by construction.
BANK_SENTS = []
if CFPB_CSV:
    col = "Consumer complaint narrative"
    df = pd.read_csv(CFPB_CSV, usecols=[col], nrows=400_000)
    seen = set()
    for t in df[col].dropna():
        t = re.sub(r"\s+", " ", str(t)).strip()
        for s in re.split(r"(?<=[.!?])\s+", t):
            s = s.strip()
            wc = len(s.split())
            if not (6 <= wc <= 45):                                    continue
            if not re.match(r"^[A-Z]", s):                             continue
            if not re.search(r"[.?!]$", s):                            continue
            if re.search(r"(?i)(http|www\.|@|x{3,}|\bxx\b|_{2,}|\|)", s): continue
            if s in seen:                                              continue
            seen.add(s); BANK_SENTS.append(s)
        if len(BANK_SENTS) > 60_000:
            break

# Held-out only. 20,000 banking sentences went into training (many as no-op
# examples), and CFPB is read here in the same order, so overlap is likely.
# Scoring the model on sentences it memorised would flatter it.
TRAIN_DIR = find_dir("/kaggle/input/**/gec_bank/data/bank_gec_train.parquet")
seen_in_train = set()
if TRAIN_DIR:
    tr = pd.read_parquet(TRAIN_DIR, columns=["input"])
    seen_in_train = {norm(x) for x in tr["input"]}
    print("train inputs:", len(seen_in_train))

before = len(BANK_SENTS)
BANK_SENTS = [s for s in BANK_SENTS if norm(s) not in seen_in_train]
print(f"banking sentences: {len(BANK_SENTS)} (dropped {before - len(BANK_SENTS)} seen in training)")
random.shuffle(BANK_SENTS)

N_EVAL = 400
EVAL_BANK_GEC, made, tries = [], 0, 0
while made < N_EVAL and tries < N_EVAL * 40:
    tries += 1
    tgt = random.choice(BANK_SENTS)
    bad = corrupt(tgt)
    if not bad:
        continue
    g = roundtrips(bad, tgt)
    if g is None or len(g) > 4:
        continue
    EVAL_BANK_GEC.append((bad, g, norm(tgt)))
    made += 1

EVAL_BANK_NOOP = random.sample(BANK_SENTS, min(N_EVAL, len(BANK_SENTS)))
print("bank gec:", len(EVAL_BANK_GEC), "| bank noop:", len(EVAL_BANK_NOOP))

In [ ]:
stage("building general eval set")

dev = pd.read_parquet(DEV_DIR).to_dict("records")
print("dev rows:", len(dev))

EVAL_GEN_GEC = []
for r in dev:
    if r["output"] == "[]":
        continue
    g = json.loads(r["output"])
    tgt = apply_edits(r["input"], g)          # gold target text
    if tgt is None:
        continue
    EVAL_GEN_GEC.append((r["input"], g, norm(tgt)))
    if len(EVAL_GEN_GEC) >= N_EVAL:
        break

EVAL_GEN_NOOP = [r["input"] for r in dev if r["output"] == "[]"][:N_EVAL]
print("general gec:", len(EVAL_GEN_GEC), "| general noop:", len(EVAL_GEN_NOOP))

## The fine-tuned model

The prompt is now built with `"\n"`, matching how LLaMA-Factory assembled every
one of the 139,625 training examples.

In [ ]:
stage("loading fine-tuned model")

from transformers import AutoTokenizer, AutoModelForCausalLM

ft_tok = AutoTokenizer.from_pretrained(MODEL_DIR)
ft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, dtype=torch.float16,
    device_map={"": 0} if torch.cuda.is_available() else None)
ft_model.eval()
if ft_tok.pad_token_id is None:
    ft_tok.pad_token = ft_tok.eos_token
ft_tok.padding_side = "left"

# LLaMA-Factory's alpaca converter joins prompt and query with "\n"
# (converter.py:100). The original eval used "\n\n" -- a mismatch on every
# single example. JOIN is a knob so the run can prove the difference.
JOIN = "\n"

@torch.no_grad()
def predict(texts, bs=32, join=None):
    j = JOIN if join is None else join
    outs = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        prompts = [ft_tok.apply_chat_template(
            [{"role": "user", "content": INSTRUCTION + j + t}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
            for t in chunk]
        enc = ft_tok(prompts, return_tensors="pt", padding=True,
                     truncation=True, max_length=512).to(ft_model.device)
        gen = ft_model.generate(**enc, max_new_tokens=160, do_sample=False,
                                pad_token_id=ft_tok.pad_token_id)
        outs += ft_tok.batch_decode(gen[:, enc["input_ids"].shape[1]:],
                                    skip_special_tokens=True)
    return outs

In [ ]:
stage("loading baseline coedit-large")

from transformers import T5ForConditionalGeneration

bl_tok = AutoTokenizer.from_pretrained("grammarly/coedit-large")
bl_model = T5ForConditionalGeneration.from_pretrained("grammarly/coedit-large")
if torch.cuda.is_available():
    bl_model = bl_model.cuda().half()
bl_model.eval()

@torch.no_grad()
def coedit_fix(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        batch = ["Fix grammatical errors in this sentence: " + t for t in texts[i:i+bs]]
        enc = bl_tok(batch, return_tensors="pt", padding=True,
                     truncation=True, max_length=256)
        enc = {k: v.to(bl_model.device) for k, v in enc.items()}
        gen = bl_model.generate(**enc, max_length=256, num_beams=1)
        out += bl_tok.batch_decode(gen, skip_special_tokens=True)
    return out

## Results

In [ ]:
stage("scoring")

def eval_gec(name, pairs):
    """Score both systems on one GEC set, raw and canonicalised."""
    srcs = [s for s, _, _ in pairs]

    ft_raw_out = predict(srcs)
    bl_out     = coedit_fix(srcs)

    rows, preds = {}, []

    # --- fine-tune -------------------------------------------------------
    acc = {"raw": [0,0,0], "canon": [0,0,0]}
    exact = valid = 0
    for (s, gold, tgt), raw in zip(pairs, ft_raw_out):
        p, ok = parse(raw); valid += ok
        p_raw   = sane(s, p)
        p_canon = canonical(s, p)
        for key, pp in (("raw", p_raw), ("canon", p_canon)):
            a, b, c = score_edits(gold, pp)
            acc[key][0] += a; acc[key][1] += b; acc[key][2] += c
        got = applied_text(s, p)
        exact += (got == tgt)
        preds.append({"set": name, "system": "qwen3-ft", "src": s,
                      "gold": json.dumps(gold, ensure_ascii=False),
                      "pred": json.dumps(p_raw, ensure_ascii=False),
                      "applied": got, "target": tgt, "raw_out": raw})
    rows["qwen3-ft"] = {
        "P_raw":   prf(*acc["raw"])[0],   "F0.5_raw":   prf(*acc["raw"])[2],
        "P_canon": prf(*acc["canon"])[0], "R_canon": prf(*acc["canon"])[1],
        "F0.5_canon": prf(*acc["canon"])[2],
        "exact_match": exact / len(pairs), "json_valid": valid / len(pairs),
    }

    # --- baseline --------------------------------------------------------
    acc = [0,0,0]; exact = 0
    for (s, gold, tgt), out in zip(pairs, bl_out):
        p = derive_edits(s, out) or []
        a, b, c = score_edits(gold, p)
        acc[0] += a; acc[1] += b; acc[2] += c
        exact += (norm(out) == tgt)
        preds.append({"set": name, "system": "coedit-large", "src": s,
                      "gold": json.dumps(gold, ensure_ascii=False),
                      "pred": json.dumps(p, ensure_ascii=False),
                      "applied": norm(out), "target": tgt, "raw_out": out})
    P, R, F = prf(*acc)
    # The baseline is canonical by construction: derive_edits IS the gold format.
    rows["coedit-large"] = {"P_raw": P, "F0.5_raw": F,
                            "P_canon": P, "R_canon": R, "F0.5_canon": F,
                            "exact_match": exact / len(pairs), "json_valid": float("nan")}
    return rows, preds


def eval_noop(pool):
    ft = predict(pool)
    bl = coedit_fix(pool)
    # Same event as the baseline: did the sentence actually change? Counting
    # "emitted a non-empty edit list" measures something else -- a no-op edit
    # like {"o":"office.","c":"office."} would score as a false positive.
    ft_flag = sum(1 for s, raw in zip(pool, ft)
                  if applied_text(s, parse(raw)[0]) != norm(s))
    bl_flag = sum(1 for a, b in zip(pool, bl) if norm(a) != norm(b))
    return {"qwen3-ft": ft_flag / len(pool), "coedit-large": bl_flag / len(pool)}


bank_rows, bank_preds = eval_gec("banking", EVAL_BANK_GEC)
gen_rows,  gen_preds  = eval_gec("general", EVAL_GEN_GEC)
stage("noop sets")
bank_noop = eval_noop(EVAL_BANK_NOOP)
gen_noop  = eval_noop(EVAL_GEN_NOOP)

In [ ]:
def table(title, rows, noop):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)
    cols = ["P_raw", "F0.5_raw", "P_canon", "R_canon", "F0.5_canon",
            "exact_match", "json_valid"]
    print(f"{'metric':<16}{'coedit-large':>16}{'qwen3-0.6b-ft':>18}")
    print("-" * 78)
    for c in cols:
        b, f = rows["coedit-large"][c], rows["qwen3-ft"][c]
        bs = "-" if b != b else f"{b:.3f}"
        print(f"{c:<16}{bs:>16}{f:>18.3f}")
    print(f"{'noop_rate':<16}{noop['coedit-large']:>16.3f}{noop['qwen3-ft']:>18.3f}")

table("BANKING  (the target domain)", bank_rows, bank_noop)
table("GENERAL  (web text)",          gen_rows,  gen_noop)

pd.DataFrame(bank_preds + gen_preds).to_csv("predictions.csv", index=False)
print("\nsaved predictions.csv")

In [ ]:
stage("recall by error type")

# Aggregate recall says the model misses errors. This says WHICH kinds it
# misses, which is the difference between "needs more data" and "needs more
# data of type X".
N_PER_TYPE = 60
by_type = {}
for fn, _w in ERRORS:
    got, tries = [], 0
    while len(got) < N_PER_TYPE and tries < N_PER_TYPE * 60:
        tries += 1
        tgt = random.choice(BANK_SENTS)
        bad = corrupt_one(tgt, fn)
        if not bad:
            continue
        gd = roundtrips(bad, tgt)
        if gd is None or len(gd) > 4:
            continue
        got.append((bad, gd, norm(tgt)))
    by_type[fn.__name__] = got

print("=" * 78)
print("RECALL BY ERROR TYPE  (banking, exactly one injected error)")
print("=" * 78)
print(f"{'error type':<18}{'n':>5}{'recall':>10}{'exact':>10}")
print("-" * 78)

rows = []
for tname, pairs in by_type.items():
    if not pairs:
        print(f"{tname:<18}{0:>5}{'-':>10}{'-':>10}")
        continue
    raws = predict([s for s, _, _ in pairs])
    tp = fnc = ex = 0
    for (src, gold, tgt), raw in zip(pairs, raws):
        p, _ = parse(raw)
        a, _b, c = score_edits(gold, canonical(src, p))
        tp += a; fnc += c
        ex += (applied_text(src, p) == tgt)
    rows.append((tname, len(pairs), tp / (tp + fnc) if tp + fnc else 0.0, ex / len(pairs)))

for tname, n, rec, ex in sorted(rows, key=lambda r: r[2]):
    print(f"{tname:<18}{n:>5}{rec:>10.3f}{ex:>10.3f}")

In [ ]:
stage("apply-strategy and difficulty")

ft_bank = [p for p in bank_preds if p["system"] == "qwen3-ft"]
ft_gen  = [p for p in gen_preds  if p["system"] == "qwen3-ft"]

print("=" * 78)
print("STRICT vs LENIENT EDIT APPLICATION")
print("=" * 78)
print(f"{'set':<12}{'strict':>10}{'lenient':>10}{'discarded':>12}{'rescued':>10}")
print("-" * 78)
for name, rows_ in (("banking", ft_bank), ("general", ft_gen)):
    st = le = disc = resc = 0
    for p in rows_:
        pred = json.loads(p["pred"])
        s, tgt = p["src"], p["target"]
        a = applied_text_strict(s, pred)
        b = applied_text(s, pred)
        st += (a == tgt); le += (b == tgt)
        if pred and apply_edits(s, sane(s, pred)) is None:
            disc += 1
            resc += (b == tgt and a != tgt)
    n = len(rows_)
    print(f"{name:<12}{st/n:>10.3f}{le/n:>10.3f}{disc:>12}{resc:>10}")

print("\n" + "=" * 78)
print("BANKING ACCURACY BY NUMBER OF GOLD EDITS")
print("=" * 78)
print(f"{'gold edits':<14}{'n':>6}{'exact':>10}")
print("-" * 78)
by_k = {}
for p in ft_bank:
    by_k.setdefault(len(json.loads(p["gold"])), []).append(p["applied"] == p["target"])
for k in sorted(by_k):
    v = by_k[k]
    print(f"{k:<14}{len(v):>6}{sum(v)/len(v):>10.3f}")

In [ ]:
stage("examples")

bad = [p for p in bank_preds if p["system"] == "qwen3-ft" and p["applied"] != p["target"]]
ok  = [p for p in bank_preds if p["system"] == "qwen3-ft" and p["applied"] == p["target"]]
print(f"banking: {len(ok)} exact / {len(ok)+len(bad)}\n")

for p in ok[:4]:
    print("SRC :", p["src"]); print("PRED:", p["pred"]); print()
print("-" * 78, "\nfailures:\n")
for p in bad[:6]:
    print("SRC :", p["src"])
    print("GOLD:", p["gold"])
    print("PRED:", p["pred"])
    print("GOT :", p["applied"])
    print("WANT:", p["target"]); print()